# Lab 3 · Lesson 2: Selenium and the Chrome DevTools Protocol
In lesson 1, Python downloaded the HTML with `requests` and read it with BeautifulSoup. That is fast and simple.
But some pages only show their data **after JavaScript runs** in a browser. For those pages, we use **Selenium**:
Python opens a real Chrome and controls it (open a page, wait, scroll, click).

A browser is **slow and heavy**, so use it only when you have to. In this lesson you will learn to:
- wait for the page to be ready (without `sleep`);
- scroll to load more products, read inside an iframe;
- use the **Chrome DevTools Protocol (CDP)**: see from Python what you see in the DevTools **Network** tab
  (the calls the page makes to the server).

All the examples use one shop: **The Souled Store** (t-shirts). Pop-ups, cookie banners and shadow DOM come in lab 4.

**Which tool should I use?**

| Where is the data? | Tool |
|---|---|
| in the HTML the server sends (even inside a `<script>` or JSON-LD) | `requests` + BeautifulSoup (lesson 1) |
| in a JSON API that the page calls | `requests` on that API (lab 2) |
| created by JavaScript, and you cannot find the API call | **Selenium** |
| only after a click, a login or a scroll | **Selenium** (but check the Network tab: there may be an API) |
| you need to prove what the user saw (screenshot) | **Selenium** |

In [ ]:
import json
import time
from urllib.parse import quote

import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

USER_AGENT = "Mozilla/5.0 (compatible; ESILV-Lab-Advanced/1.0; educational use)"
SOULED = "https://www.thesouledstore.com/men/t-shirts"

## Set up the driver: two modes
**You need:** Google Chrome, and the `selenium` package (installed by `uv sync`). The first time, Selenium downloads
the `chromedriver` that matches your Chrome by itself: nothing else to install.

**Browser mode**: a Chrome window opens, you watch everything.

```python
driver = webdriver.Chrome()                      # a window opens
driver.get("https://www.thesouledstore.com/men/t-shirts")
driver.quit()                                    # always close it
```

**Headless mode**: the same Chrome, without a window (for real scraping).

```python
options = webdriver.ChromeOptions()
options.add_argument("--headless=new")           # the only difference
driver = webdriver.Chrome(options=options)
driver.get("https://www.thesouledstore.com/men/t-shirts")
driver.quit()
```

Everything after that (`get`, `find_element`, `quit` ...) is the same code in both modes.

## 0. First, watch the browser
Run the next cell and look at your screen: a **Chrome window opens by itself**, goes to the shop, and closes.
That is Selenium: Python is driving a real browser, like a person would.

In [ ]:
start = time.time()
visible_driver = webdriver.Chrome()                  # a Chrome WINDOW opens
visible_driver.get(SOULED)                           # Python types the URL and waits for the page
load_seconds = time.time() - start
print("page title:", visible_driver.title)
print(f"window opened + page loaded in {load_seconds:.1f} s")

time.sleep(5)                                        # only so you have time to watch (never in real scraping)
visible_driver.quit()                                # the window closes

Nice to watch! For real scraping, the window is a waste: drawing it takes time and memory, nobody looks at it, and
a server has no screen at all. That is why Chrome has a **headless** mode: the same browser, the same pages, the same
JavaScript ... but **without a window**.

## 1. Start Chrome: with or without a window
One switch decides, `HEADLESS`:
- `HEADLESS = False`: a Chrome **window** opens and you **watch** every step (scroll, clicks, banners). We use this in class.
- `HEADLESS = True`: **no window**, the same code runs in the background. Use this for real scraping.

Try both: run the notebook once with `False`, then change it to `True`, run again ... same results, no window.

Selenium 4 finds (and downloads, the first time) the `chromedriver` that matches your Chrome: nothing to install.

| Option | Why |
|---|---|
| `--headless=new` | no window (only when `HEADLESS = True`) |
| `--window-size=1400,1000` | pages adapt to the size: a small window can show the mobile site |
| `--user-agent=...` | our honest robot name (headless Chrome says `HeadlessChrome` otherwise) |
| `goog:loggingPrefs` `performance` | Chrome keeps its network events for us (section 6) |

In [ ]:
HEADLESS = False        # False = you SEE the Chrome window (class demo). True = no window (real scraping).


def make_driver(headless=HEADLESS):
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,1000")
    options.add_argument(f"--user-agent={USER_AGENT}")
    options.set_capability("goog:loggingPrefs", {"performance": "ALL"})
    return webdriver.Chrome(options=options)


start = time.time()
driver = make_driver()                               # every example below uses this driver
driver.get(SOULED)
load_seconds = time.time() - start
mode = "headless (no window)" if HEADLESS else "with a window"
print("page title:", driver.title)
print(f"Chrome {mode}: started + page loaded in {load_seconds:.1f} s")
print("user agent:", driver.execute_script("return navigator.userAgent"))

> ⚠️ Every `make_driver()` starts a Chrome (~200 MB). **Always** `driver.quit()` at the end (last cell). In a
> program: `try: ... finally: driver.quit()`, so Chrome closes even after an error.

## 2. Find elements
`driver.find_element(By.CSS_SELECTOR, ...)` returns ONE element (or raises `NoSuchElementException`);
`find_elements` returns a list (maybe empty). A `WebElement` has `.text` (the **visible** text),
`.get_attribute("href")` (absolute URL!), `.get_property(...)`, and you can search inside it.

In [ ]:
cards = driver.find_elements(By.CSS_SELECTOR, ".productCard")
print(len(cards), "cards right now")
if cards:
    print(repr(cards[0].text))
    print(cards[0].find_element(By.CSS_SELECTOR, "a").get_attribute("href"))

Maybe 0, maybe 24: it depends on how fast the page was. Never trust "it worked on my machine".

## 3. Wait: explicitly, never `sleep`
| | |
|---|---|
| `time.sleep(5)` | too long when the page is fast, too short when it is slow. ❌ |
| `driver.implicitly_wait(5)` | every `find_element` waits up to 5 s. Hides problems, slows `find_elements`. ❌ |
| `WebDriverWait(driver, 10).until(condition)` | waits **exactly** until the condition is true (checks every 0.5 s). ✅ |

Ready-made conditions: `EC.presence_of_element_located`, `EC.visibility_of_element_located`,
`EC.element_to_be_clickable`, `EC.invisibility_of_element_located` ... Or **your own**: any function of the driver.

In [ ]:
wait = WebDriverWait(driver, 15)

# 1. a ready-made condition: wait until the first card is VISIBLE
first_card = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, ".productCard")))
print("first card:", first_card.text.replace("\n", " | "))


# 2. your own condition: a normal function that receives the driver and returns True or False
def at_least_24_cards(driver):
    return len(driver.find_elements(By.CSS_SELECTOR, "[data-impression]")) >= 24


wait.until(at_least_24_cards)          # give the function's NAME (no brackets): Selenium calls it every 0.5 s
print(len(driver.find_elements(By.CSS_SELECTOR, "[data-impression]")), "cards")


# 3. a condition that never becomes True: after 2 s, TimeoutException
def at_least_1000_cards(driver):
    return len(driver.find_elements(By.CSS_SELECTOR, "[data-impression]")) >= 1000


try:
    WebDriverWait(driver, 2).until(at_least_1000_cards)
except TimeoutException:
    print("TimeoutException after 2 s: never 1000 cards (that is how a wait says no)")

## 4. Infinite scroll
Scroll to the bottom, then wait until **more** cards are there. No new card within a few seconds:
the end of the list. Always a maximum number of scrolls.

The trap (try it: remove the "up one screen" line and the `sleep` after it): this page loads more when you **enter** its bottom zone.
Just after new cards arrive you are still in it, so "scroll to the bottom" again does nothing. Go up one screen,
then down: like a human.

`driver.execute_script("...")` runs a small piece of **JavaScript inside the page**. Here it scrolls, like the mouse wheel. Try the same line by hand: DevTools → **Console** tab → type `window.scrollTo(0, document.body.scrollHeight)` → Enter: the page jumps to the bottom.

In [ ]:
def more_cards_than_before(driver):
    return len(driver.find_elements(By.CSS_SELECTOR, "[data-impression]")) > cards_before


history = [len(driver.find_elements(By.CSS_SELECTOR, "[data-impression]"))]
for _ in range(3):
    cards_before = history[-1]
    driver.execute_script("window.scrollBy(0, -window.innerHeight)")      # up one screen
    time.sleep(0.3)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")   # down to the bottom
    try:
        WebDriverWait(driver, 8).until(more_cards_than_before)
    except TimeoutException:
        break                                                              # no new cards: end of the list
    history.append(len(driver.find_elements(By.CSS_SELECTOR, "[data-impression]")))
print("cards after each scroll:", history)

## 5. `page_source` → BeautifulSoup
`driver.page_source` is the HTML **now**, after JavaScript. Parse it with BeautifulSoup, like lesson 1: one big
string, fast, no back-and-forth with Chrome (each `element.text` is a round trip to the browser).

In [ ]:
soup = BeautifulSoup(driver.page_source, "lxml")
for box in soup.select("[data-impression]")[:3]:
    print(box["data-impression"], "|", box.select_one(".product-name-text").get_text(strip=True),
          "|", box.select_one(".offer").get_text(strip=True))

## 6. CDP: spy on the page's network calls
When you scroll, new t-shirts appear. Where do they come from? The page **asks the server** for them, in the
background. You can see these calls in DevTools → **Network** tab. The **Chrome DevTools Protocol (CDP)** lets
Python read the same list.

**Try it by hand first:** DevTools → Network tab → in the filter box type `listing` → scroll down the page.
Each scroll adds a line named `listing`, method **POST**. Click it:
- **Payload** = what the page sent to the server (`"page": 2`, `"page": 3` ...);
- **Response** = what the server answered: the products, in JSON.

**Python does the same:** `driver.get_log("performance")` = everything Chrome wrote in its Network log (we
switched it on in section 1 with `goog:loggingPrefs`). We keep only the requests whose URL contains `products/listing`.

In [ ]:
# read Chrome's Network log and keep only the POST requests to the product list
log_entries = driver.get_log("performance")
print(len(log_entries), "network events in the log")

api_calls = []
for entry in log_entries:
    event = json.loads(entry["message"])["message"]          # each entry is a JSON text
    if event["method"] == "Network.requestWillBeSent":        # = "Chrome is sending a request"
        request = event["params"]["request"]
        if "products/listing" in request["url"] and request["method"] == "POST":
            api_calls.append(event["params"])
            print(request["method"], request["url"])
            print("   sent:", request.get("postData"))         # the Payload: which page of products

## 7. ... and back to `requests`
In section 6 we found the request the page sends when you scroll: its address (`.../products/listing`) and what it
sends (`"page": 2` ...). We can send the **same request ourselves** with `requests`. No browser, no scrolling.
And we can ask for **100 products at once** instead of 24.

Remember: the browser was only useful to **find** this hidden request. Once you know it, `requests` is enough.

In [ ]:
body = {"category": "t-shirts", "tags": "", "artist": "", "filters": {"sort": "Relevance"}, "page": 1, "offset": 100,
        "tiptile": 1, "is_exclusive": 0, "gender_type": 3}
response = requests.post("https://api.thesouledstore.com/api/v2/pre-order/products/listing", json=body,
                         headers={"User-Agent": USER_AGENT}, timeout=20)
data = response.json()
print(response.status_code, len(data["products"]), "products in ONE call, out of", data["numberOfProducts"])

## 8. iframes
An `<iframe>` is a **page inside the page** (a YouTube video, a payment form, a reviews box).
Selenium looks at only **one page at a time**: it does not see what is inside the iframe.
- `driver.switch_to.frame(...)`: go **into** the iframe;
- `driver.switch_to.default_content()`: come **back** to the main page.

To keep it simple, we do not use a real shop here: we write a tiny page with one iframe ourselves.

In [ ]:
# a tiny page written by us: a title, and an iframe (a page inside the page) with one review in it
page = """
<h1>Outside</h1>

<iframe id="reviews"
        srcdoc="<p class='review'>Great phone, 5/5</p>">
</iframe>
"""
driver.get("data:text/html;charset=utf-8," + quote(page))
print("from outside:", driver.find_elements(By.CSS_SELECTOR, ".review"))
driver.switch_to.frame(driver.find_element(By.ID, "reviews"))
print("inside      :", driver.find_element(By.CSS_SELECTOR, ".review").text)
driver.switch_to.default_content()
print("back        :", driver.find_element(By.TAG_NAME, "h1").text)

In [ ]:
driver.quit()